In [1]:
# !pip install folium

In [2]:
# !pip install requests

In [3]:
import numpy as np
import pandas as pd
import folium
import json

In [4]:

# json 파일의 경로들
crime_path = "../crime/crime.json"  
population_path = "../population/population.json"


crime_json = json.load(open(crime_path,"r", encoding = 'utf-8'))
population_json = json.load(open(population_path,"r",encoding='utf-8'))

crime_df = pd.DataFrame(crime_json)
population_df = pd.DataFrame(population_json)

In [5]:
crime_df

,sido,sigungu,crime,violence
0,서울특별시,중구,198,1378
1,서울특별시,용산구,282,1941
2,서울특별시,성동구,128,1033
3,서울특별시,광진구,252,1303
4,서울특별시,동대문구,144,1462
...,...,...,...,...
94,경기도,의왕시,43,363
95,경기도,의정부시,201,2129
96,경기도,이천시,91,960
97,경기도,파주시,196,2058


In [6]:
population_df

,sido,sigungu,population
0,전국,소계,51217221
1,서울특별시,소계,9331828
2,서울특별시,종로구,138336
3,서울특별시,중구,120544
4,서울특별시,용산구,203854
...,...,...,...
242,경상남도,거창군,59588
243,경상남도,합천군,40225
244,제주특별자치도,소계,670368
245,제주특별자치도,제주시,488348


In [7]:
# crime_df.join(population_df) -> 에러 발생 인덱스 기준으로 옆으로 붙이는 함수임. 겹치는 컬럼이 있어서 에러가 발생한것
# 이 경우는 merge를 선택하는 것이 좋음

merge_df = pd.merge(crime_df, population_df) # -> 이렇게 할 경우 기본적으로 inner join임.두 키가 정확히 일치하는 행만 남게됨.



In [8]:
merge_df

,sido,sigungu,crime,violence,population
0,서울특별시,중구,198,1378,120544
1,서울특별시,용산구,282,1941,203854
2,서울특별시,성동구,128,1033,273669
3,서울특별시,광진구,252,1303,331963
4,서울특별시,동대문구,144,1462,338735
...,...,...,...,...,...
94,경기도,의왕시,43,363,154488
95,경기도,의정부시,201,2129,461271
96,경기도,이천시,91,960,222443
97,경기도,파주시,196,2058,511308


In [9]:
merge_df.dtypes # 현재 population 데이터타입이 object로 돼 있음 -> 이걸 int로 바꿔줘야됨

sido          object
sigungu       object
crime          int64
violence       int64
population    object
dtype: object

In [10]:
merge_df['sido'] = merge_df['sido'].astype(str)
merge_df['sigungu'] = merge_df['sigungu'].astype(str)
merge_df['population'] =merge_df['population'].astype(int)

In [11]:
type(crime_df.loc[0,'sido'])

str

In [12]:
# 강력범죄
merge_df['crime_rate'] = merge_df['crime']/ merge_df['population']*10000

merge_df['violence_rate'] = merge_df['violence'] / merge_df['population']*10000

# 전체 범죄율 혹시나 나중에 쓰일까봐?
merge_df["total_rate"] = (merge_df["crime"] + merge_df["violence"]) / merge_df["population"] * 100000

In [13]:
geo_path = "geojson_data/hangjeongdong_서울특별시.geojson"


with open(geo_path, "r", encoding="utf-8") as f:
    geo = json.load(f)

geo["features"][0]["properties"]

## 밑에 결과를 보면 종로구가 sggnm이라고 돼있음
# 이건 꼭 지켜줘야됨
# 내가 보기편하려고 gu_name으로 바꿨다가 아무것도 안 떴었음!

{'OBJECTID': 1,
 'adm_nm': '서울특별시 종로구 사직동',
 'adm_cd': '1101053',
 'adm_cd2': '1111053000',
 'sgg': '11110',
 'sido': '11',
 'sidonm': '서울특별시',
 'sggnm': '종로구'}

In [14]:

### geojson에 현재 구 개수가 몇 개가 있는지 확인하는 코드
### 딱히 필요는 없음!! 궁금해서 gpt한테 물어봄
sgg_list = []

for feat in geo["features"]:
    # sggnm은 정해져 있는 이름임!
    sgg_list.append(feat["properties"].get("sggnm"))

unique_sgg = sorted(set(sgg_list))
print("구 개수:", len(unique_sgg))
print(unique_sgg)

구 개수: 25
['강남구', '강동구', '강북구', '강서구', '관악구', '광진구', '구로구', '금천구', '노원구', '도봉구', '동대문구', '동작구', '마포구', '서대문구', '서초구', '성동구', '성북구', '송파구', '양천구', '영등포구', '용산구', '은평구', '종로구', '중구', '중랑구']


In [15]:
# 서울 데이터만 뽑기
seoul_df = merge_df[merge_df["sido"] == "서울특별시"].copy()

# 만명당 총범죄율 계산
seoul_df["total_rate_1man"] = (seoul_df["crime"] + seoul_df["violence"]) / seoul_df["population"] * 10000 

# Choropleth에서 연결할 key
# json 데이터 쪽에서 key를 만들었으니까 마찬가지로
seoul_df["join_key"] = seoul_df["sigungu"]

# GeoJson에도 key를 만들어줘야 연결이 됨
# 지도쪽(GeoJSON)에도 “지역 이름 키”가 있어야 하고
# 데이터쪽(DataFrame)에도 “같은 지역 이름 키”가 있어야 함
# → 둘이 같은 키로 연결해서 색칠함


# features : geo의 각 구역을 하나씩 돌기
for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")



#### 서울 중심 지도 테스트

In [16]:
# 서울 중심 근처로 지도 생성 x테스트
m = folium.Map(
    location=[37.55, 126.98],   # 서울 중심쯤
    zoom_start=11,              # 확대 정도
    tiles="cartodbpositron"     # 배경 스타일(깔끔한 회색 지도)
)

m

## 여기 밑에 지도가 안 뜬다면 
# 설정해야될 것들
# ctrl + shift + p 누르면 위에 검색창이 뜨는데 거기에 trust를 입력 - > Workspaces: Manage Workspace Trust
# 그리고 밑에 add folder가 있는데 현재 이 프로젝트를 하는 폴더를 추가
# 그리고 현재 이 crime_rate.ipynb를 저장을 하고 껐다가 다시 열기

#### 서울 지도 경계 확인

In [17]:
# (1) 지도 위에  색으로 칠하는 레이어 추가
folium.Choropleth(
    geo_data=geo,                       # 지도 경계
    data=seoul_df,                      # 서울 데이터(구별 범죄율)
    columns=["join_key", "total_rate_1man"],  # join_key로 연결해서 total_rate_1man 값으로 색칠
    key_on="feature.properties.join_key",     # GeoJSON 안의 join_key 위치

    fill_color="YlOrRd",                # 색상(노랑~빨강)

    fill_opacity=0.75,                  # 색 투명도

    line_opacity=0.2,                   # 경계선 투명도

    nan_fill_opacity=0.05,              # 매칭 실패 지역은 거의 투명
    
    legend_name="서울 총 범죄율 (건/만명)"     # 상단에 서울 총 범죄율 (건/만명) 표시
).add_to(m)


# 지도 출력
m


#### 경기도

In [ ]:

## 여기서 부터 설명은 위 서울특별시에서 작성했던 코드와 같음

geo_path = "geojson_data/hangjeongdong_경기도.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '경기도'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 경기도 중심쯤 좌표
m = folium.Map(location=[37.4, 127.1], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="Blues",
    fill_opacity= 0.75,
    line_opacity= 0.002,
    nan_fill_opacity= 0.05,
    legend_name="경기도 총 범죄율 (건/만명)"
).add_to(m)

m

## 광주 광역시 지도

In [28]:
geo_path = "geojson_data/hangjeongdong_광주광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '광주광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 경기도 중심쯤 좌표
m = folium.Map(location=[35.16, 126.85], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="YlOrRd",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="광주광역시 총 범죄율 (건/만명)"
).add_to(m)

m

### 대구 광역시


In [27]:

geo_path = "geojson_data/hangjeongdong_대구광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '대구광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 경기도 중심쯤 좌표
m = folium.Map(location=[35.87, 128.60], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="YlOrRd",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="대구광역시 총 범죄율 (건/만명)"
).add_to(m)

m

### 대전 광역시

In [26]:
geo_path = "geojson_data/hangjeongdong_대전광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '대전광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 대전광역시 중심쯤 좌표
m = folium.Map(location=[36.3504, 127.3845], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="Greens",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="대전광역시 총 범죄율 (건/만명)"
).add_to(m)

m

### 부산 광역시

In [22]:
geo_path = "geojson_data/hangjeongdong_부산광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '부산광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 부산광역시 중심쯤 좌표
m = folium.Map(location=[35.1796, 129.0756], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="Reds",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="부산광역시 총 범죄율 (건/만명)"
).add_to(m)

m

### 울산 광역시

In [25]:
geo_path = "geojson_data/hangjeongdong_울산광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '울산광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 울산 광역시 중심쯤 좌표
m = folium.Map(location=[35.5384, 129.3114], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="YlGnBu",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="울산광역시 총 범죄율 (건/만명)"
).add_to(m)

m

### 인천 광역시

In [24]:
geo_path = "geojson_data/hangjeongdong_인천광역시.geojson"
with open(geo_path,'r',encoding='utf-8') as f:
    geo = json.load(f)


gyeonggi_df = merge_df[merge_df['sido'] == '인천광역시'].copy()


gyeonggi_df['total_rate_1man'] = (gyeonggi_df['violence'] + gyeonggi_df['crime'])/ gyeonggi_df['population'] *10000

gyeonggi_df["join_key"] = gyeonggi_df["sigungu"]

for feat in geo["features"]:
    feat["properties"]["join_key"] = feat["properties"].get("sggnm")


# 인천 중심쯤 좌표
m = folium.Map(location=[37.4563, 126.7052], zoom_start=9, tiles="cartodbpositron")


folium.Choropleth(
    geo_data= geo,
    data = gyeonggi_df,
    columns= ['join_key','total_rate_1man'],
    key_on="feature.properties.join_key",
    fill_color="PuBu",
    fill_opacity= 0.75,
    line_opacity= 0.2,
    nan_fill_opacity= 0.05,
    legend_name="인천광역시 총 범죄율 (건/만명)"
).add_to(m)

m